# Data Acquisition

In [ ]:
import os
import cv2
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import transforms
import matplotlib.pyplot as plt
import logging

# Membisukan peringatan library agar log bersih
os.environ['OPENCV_LOG_LEVEL'] = 'OFF'
logging.getLogger("libtiff").setLevel(logging.ERROR)

# =================================================================
# 1. KONFIGURASI JALUR DATA (PATH CONFIGURATION)
# =================================================================
BASE_PATH = Path("/kaggle/input/datasets/dimaspashaakrilian/openearthmap/OpenEarthMap_wo_xBD")

# Validasi jalur dataset
if not BASE_PATH.exists():
    search_results = list(Path("/kaggle/input").rglob("OpenEarthMap_wo_xBD"))
    if search_results:
        BASE_PATH = search_results[0]
        print(f"Jalur dataset ditemukan di: {BASE_PATH}")
    else:
        print("Kesalahan: Folder OpenEarthMap_wo_xBD tidak ditemukan.")

# =================================================================
# 2. FUNGSI PEMETAAN FILE
# =================================================================
def get_paths(split_name):
    txt_file = BASE_PATH / f"{split_name}.txt"
    if not txt_file.exists():
        print(f"File {split_name}.txt tidak ditemukan.")
        return [], []

    with open(txt_file, 'r') as f:
        filenames = [line.strip().replace('.tif', '') for line in f.readlines()]
    
    img_paths, mask_paths = [], []
    all_tifs = list(BASE_PATH.glob("**/images/*.tif"))
    file_map = {f.stem: f.parent.parent for f in all_tifs}
    
    for name in filenames:
        pure_name = Path(name).stem
        if pure_name in file_map:
            city_dir = file_map[pure_name]
            img_paths.append(str(city_dir / "images" / f"{pure_name}.tif"))
            mask_paths.append(str(city_dir / "labels" / f"{pure_name}.tif"))
    
    return img_paths, mask_paths

# =================================================================
# 3. KELAS DATASET (OPTIMIZED FOR EM/GMM)
# =================================================================
class OpenEarthMapDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load Image
        image = cv2.imread(self.img_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Load Mask
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        
        # Konversi label 1-8 menjadi 0-7 (untuk klasifikasi)
        mask = mask.astype(np.int64) - 1
        mask[mask < 0] = 0 
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        return image, mask

# Transformasi: Ukuran 512x512 lebih efisien untuk clustering piksel
# Normalisasi diatur ke standard [0,1] agar EM lebih mudah mengolah distribusi warna
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=(0, 0, 0), std=(1, 1, 1)), # Skala 0-1
    transforms.ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0, 0, 0), std=(1, 1, 1)), # Skala 0-1
    transforms.ToTensorV2()
])

# =================================================================
# 4. EKSEKUSI DATA LOADER
# =================================================================
train_imgs, train_masks = get_paths("train")
val_imgs, val_masks = get_paths("val")

if len(train_imgs) > 0:
    train_ds = OpenEarthMapDataset(train_imgs, train_masks, transform=train_transform)
    val_ds = OpenEarthMapDataset(val_imgs, val_masks, transform=val_transform)
    
    # Batch size bisa lebih besar karena tidak ada overhead memori dari DINOv2
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
    
    # Verifikasi batch
    images, masks = next(iter(train_loader))
    print(f"Data Acquisition Berhasil!")
    print(f"Jumlah Training: {len(train_imgs)}")
    print(f"Dimensi Tensor Image: {images.shape}") # [Batch, 3, 512, 512]
    print(f"Dimensi Tensor Mask: {masks.shape}")   # [Batch, 512, 512]
    
    # Visualisasi untuk pengecekan
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(images[0].permute(1,2,0).numpy())
    plt.title("Input RGB (Normalised)")
    plt.subplot(1, 2, 2)
    plt.imshow(masks[0].numpy(), cmap='terrain')
    plt.title("Target Mask")
    plt.show()
else:
    print("Data tidak ditemukan. Pastikan BASE_PATH sudah benar.")

# Preprocessing & Feature Engineering

In [ ]:
from sklearn.preprocessing import StandardScaler

def extract_features(images):
    """
    Mengubah batch gambar menjadi matriks fitur (Pixels, Features).
    Fitur: R, G, B, ExG (Excess Green), ExR (Excess Red), X, Y.
    """
    # 1. Konversi ke Numpy: [Batch, C, H, W] -> [Batch, H, W, C]
    img_np = images.permute(0, 2, 3, 1).numpy()
    b, h, w, c = img_np.shape
    
    # Ambil tiap channel warna
    R = img_np[:, :, :, 0]
    G = img_np[:, :, :, 1]
    B = img_np[:, :, :, 2]

    # 2. Feature Engineering: Indeks Warna (Vegetasi & Tanah)
    # Excess Green (ExG): Membantu membedakan tanaman hijau
    # Formula: 2*G - R - B
    exg = 2*G - R - B
    
    # Excess Red (ExR): Membantu membedakan bangunan/tanah terbuka
    # Formula: 1.4*R - G
    exr = 1.4*R - G

    # 3. Feature Engineering: Koordinat Spasial (X, Y)
    # Membuat grid koordinat yang dinormalisasi (0 sampai 1)
    x_coords, y_coords = np.meshgrid(np.linspace(0, 1, w), np.linspace(0, 1, h))
    x_coords = np.tile(x_coords, (b, 1, 1))
    y_coords = np.tile(y_coords, (b, 1, 1))

    # 4. Stack semua fitur menjadi satu matriks
    # Dimensi akhir: (Batch * H * W, 7 fitur)
    features = np.stack([R, G, B, exg, exr, x_coords, y_coords], axis=-1)
    features_flattened = features.reshape(-1, 7)

    # 5. Scaling: Agar fitur koordinat dan warna punya bobot yang setara
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features_flattened)
    
    print(f"Fitur berhasil diekstrak. Dimensi matriks: {features_scaled.shape}")
    return features_scaled, (b, h, w)

# Ambil satu batch dari Data Loader sebelumnya
images, masks = next(iter(train_loader))

# Jalankan Preprocessing
X_features, spatial_dim = extract_features(images)

# Reshaping Data

In [ ]:
def prepare_for_em(features_scaled, spatial_dim):
    """
    Menyiapkan data yang sudah di-scale untuk masuk ke algoritme EM.
    """
    b, h, w = spatial_dim
    
    # 1. Pastikan data berbentuk (N_Samples, N_Features)
    # Di tahap sebelumnya, kita sudah melakukan flatten menjadi:
    # (Batch * Height * Width, Jumlah_Fitur)
    X_em = features_scaled
    
    print(f"Data siap untuk EM!")
    print(f"Total piksel (baris): {X_em.shape[0]}")
    print(f"Total fitur (kolom): {X_em.shape[1]}")
    
    return X_em

# Menjalankan fungsi penyiapan
X_input_em = prepare_for_em(X_features, spatial_dim)

# Algoritme EM (GMM)

In [ ]:
from sklearn.mixture import GaussianMixture
import time

def train_em_gmm(X_input, n_clusters=8):
    """
    Menjalankan algoritme EM untuk mengelompokkan tutupan lahan.
    """
    # 1. Inisialisasi Model
    # covariance_type='full' memungkinkan tiap cluster memiliki bentuk elips yang unik
    gmm = GaussianMixture(
        n_components=n_clusters, 
        covariance_type='full', 
        max_iter=100, 
        random_state=42,
        verbose=1 # Menampilkan log iterasi EM
    )

    print(f"Memulai iterasi EM untuk {n_clusters} kelas tutupan lahan...")
    start_time = time.time()

    # 2. Fit Model & Predict (Proses E-Step dan M-Step terjadi di sini)
    # labels: hasil "hard clustering"
    labels = gmm.fit_predict(X_input)
    
    # 3. Ambil Probabilitas (Soft Clustering)
    # probabilities: hasil "soft clustering" untuk analisis mixed pixels
    probabilities = gmm.predict_proba(X_input)

    end_time = time.time()
    print(f"Proses EM selesai dalam {end_time - start_time:.2f} detik.")
    print(f"Apakah model konvergen? {gmm.converged_}")
    
    return labels, probabilities, gmm

# Eksekusi Pelatihan
n_classes = 8 # Sesuai dengan kelas di OpenEarthMap
labels, probs, model_gmm = train_em_gmm(X_input_em, n_clusters=n_classes)

# Post-Processing & Re-mapping

In [ ]:
def post_process_results(labels, probs, spatial_dim):
    """
    Mengembalikan hasil EM ke dimensi spasial dan menghitung peta probabilitas.
    """
    b, h, w = spatial_dim
    n_clusters = probs.shape[1]

    # 1. Reshape Label (Hard Clustering)
    # Dari (B*H*W,) kembali ke (B, H, W)
    labels_spatial = labels.reshape(b, h, w)
    
    # 2. Reshape Probabilitas (Soft Clustering)
    # Mengambil probabilitas tertinggi sebagai tingkat keyakinan (Confidence Map)
    conf_map = np.max(probs, axis=1).reshape(b, h, w)
    
    # 3. Probability Maps untuk tiap kelas (Optional)
    # Dimensi: (B, H, W, N_Clusters)
    probs_spatial = probs.reshape(b, h, w, n_clusters)
    
    return labels_spatial, conf_map, probs_spatial

# Eksekusi Reshaping
labels_map, confidence, all_probs = post_process_results(labels, probs, spatial_dim)

print(f"Reshaping selesai! Bentuk peta: {labels_map.shape}")

In [ ]:
def plot_final_comparison(images, masks, labels_map, confidence, batch_idx=0):
    plt.figure(figsize=(20, 5))
    
    # 1. Original Image (Denormalisasi)
    img_show = images[batch_idx].permute(1, 2, 0).numpy()
    img_show = (img_show * 1.0).clip(0, 1) # Skala 0-1
    
    plt.subplot(1, 4, 1)
    plt.imshow(img_show)
    plt.title("Original RGB")
    plt.axis('off')

    # 2. Ground Truth
    plt.subplot(1, 4, 2)
    plt.imshow(masks[batch_idx], cmap='terrain')
    plt.title("Ground Truth (Manual)")
    plt.axis('off')

    # 3. EM Clustering Result
    # Ingat: Warna mungkin berbeda karena urutan cluster di EM bersifat acak
    plt.subplot(1, 4, 3)
    plt.imshow(labels_map[batch_idx], cmap='tab10')
    plt.title("EM (GMM) Result")
    plt.axis('off')

    # 4. Confidence Map (Semakin terang, semakin yakin EM-nya)
    plt.subplot(1, 4, 4)
    plt.imshow(confidence[batch_idx], cmap='magma')
    plt.title("EM Confidence Score")
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Tampilkan hasil untuk satu sampel di batch
plot_final_comparison(images, masks, labels_map, confidence, batch_idx=0)

# Evaluasi

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix, jaccard_score, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# =================================================================
# 1. LABEL ALIGNMENT (Mencocokkan Cluster EM ke Ground Truth)
# =================================================================
def evaluate_em_performance(gt_masks, pred_labels, n_clusters=8):
    # Flatten data untuk komputasi matriks
    gt_flatten = gt_masks.flatten()
    pred_flatten = pred_labels.flatten()
    
    # Hitung Confusion Matrix sebagai cost matrix untuk Hungarian Algorithm
    cm = confusion_matrix(gt_flatten, pred_flatten, labels=range(n_clusters))
    
    # Mencari pasangan label terbaik (maximize diagonal/agreement)
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {col: row for row, col in zip(row_ind, col_ind)}
    
    # Terapkan mapping ke hasil prediksi
    aligned_preds_flatten = np.array([mapping[l] for l in pred_flatten])
    aligned_preds = aligned_preds_flatten.reshape(pred_labels.shape)
    
    # =================================================================
    # 2. PERHITUNGAN METRIK
    # =================================================================
    # Mean IoU (mIoU)
    iou_per_class = jaccard_score(gt_flatten, aligned_preds_flatten, average=None, labels=range(n_clusters))
    miou = np.mean(iou_per_class)
    
    # Pixel Accuracy
    accuracy = np.mean(gt_flatten == aligned_preds_flatten)
    
    print("===== HASIL EVALUASI ALGORITME EM =====")
    print(f"Overall Pixel Accuracy : {accuracy:.4f}")
    print(f"Mean IoU (mIoU)        : {miou:.4f}")
    print(f"Model Converged        : {model_gmm.converged_}")
    print(f"BIC Score              : {model_gmm.bic(X_input_em):.2f}")
    print("-" * 39)
    
    # Menampilkan IoU per kelas
    openearthmap_labels = ['Bareland', 'Grass', 'Pavement', 'Road', 'Tree', 'Water', 'Cropland', 'Building']
    for i, score in enumerate(iou_per_class):
        label_name = openearthmap_labels[i] if i < len(openearthmap_labels) else f"Class {i}"
        print(f"{label_name:15} | IoU: {score:.4f}")
        
    return aligned_preds, cm, mapping

# Jalankan Evaluasi
aligned_labels, conf_matrix, final_mapping = evaluate_em_performance(masks.numpy(), labels_map, n_clusters=n_classes)

# =================================================================
# 3. VISUALISASI AKHIR
# =================================================================
plt.figure(figsize=(20, 6))

# Plot 1: Confusion Matrix
plt.subplot(1, 3, 1)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix (Original Clusters)")
plt.xlabel("EM Clusters")
plt.ylabel("Ground Truth")

# Plot 2: Ground Truth Map
plt.subplot(1, 3, 2)
plt.imshow(masks[0].numpy(), cmap='terrain')
plt.title("Ground Truth Mask")
plt.axis('off')

# Plot 3: Aligned EM Prediction Map
plt.subplot(1, 3, 3)
plt.imshow(aligned_labels[0], cmap='terrain') # Menggunakan cmap yang sama dengan GT
plt.title("EM Aligned Prediction")
plt.axis('off')

plt.tight_layout()
plt.show()